**PS3 — Root Cause Classification**

---
SageMaker Studio | MLflow | Feature Store | Model Registry | Model Monitor

Architecture  : CUBIC MARS AWS v6
Targets       : failure_level_label (3-class: MINOR/MAJOR/CRITICAL)
                root_cause_category (8-class: rule-based text derivation)
Grain         : one row per ServiceNow availability event per device
Models        : XGBoost multiclass + LightGBM + CatBoost
                NLP pipeline (TF-IDF on fault text) + Combined structured+NLP
Feasibility   : 70%  (30% gap — SVN_STAGE=0 rows; AE_FAILURE_LEVEL Ventra taxonomy confirmed 2026-06-23)
                AE_FAILURE_LEVEL (hardware faults): 1=NONPAYMENT, 2=PURCHASE_CARD,
                  3=PURCHASE_PRODUCT, 4=ALL_PURCHASE, 5=ALL_FUNCTIONS, 16=BUS_READER_ASSEMBLY
                Decoded via silver.dim_failure_level (S18). Updated from 55% → 70%.
Data source   : CTA.SERVICENOW_AVAILABILITY_EVENTS (295,960 rows) — simplified target

---

In [ ]:
# ── CELL 1 : Install ──────────────────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "s3fs","pyarrow",
    "scikit-learn>=1.4","xgboost>=2.0","lightgbm>=4.3","catboost>=1.2",
    "optuna>=3.6","shap>=0.45","mlflow>=2.13", "sagemaker-mlflow",
    "boto3","sagemaker>=2.220,<3.0","pandas>=2.0","matplotlib","seaborn","nltk",
]
subprocess.check_call([sys.executable,"-m","pip","install","-q","--upgrade"]+pkgs)
import nltk; nltk.download("stopwords",quiet=True); nltk.download("wordnet",quiet=True)
print("OK")

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────────────
import os, json, time, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap, optuna, boto3, sagemaker, joblib, tarfile

# deltalake removed: gold tables exported as Parquet; pd.read_parquet() used in CELL 4
from sklearn.model_selection         import TimeSeriesSplit, StratifiedKFold
from sklearn.preprocessing           import LabelEncoder, label_binarize
from sklearn.pipeline                import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model            import LogisticRegression
from sklearn.ensemble                import RandomForestClassifier
from sklearn.base                    import BaseEstimator, TransformerMixin
from sklearn.metrics                 import (roc_auc_score, f1_score,
                                             classification_report,
                                             confusion_matrix)
from xgboost   import XGBClassifier
from lightgbm  import LGBMClassifier, early_stopping, log_evaluation
from catboost  import CatBoostClassifier

import mlflow, mlflow.sklearn
from mlflow.models.signature import infer_signature
try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session                       # sagemaker >= 3
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role            # sagemaker >= 3
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Imports OK")

In [ ]:
# ── CELL 3 : Configuration ────────────────────────────────────────────────────
ARTIFACT_BUCKET     = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
GOLD_BUCKET         = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX         = "chicago/gold/device_ps3_incident"
GOLD_S3             = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION          = Session()
ROLE                = get_execution_role()
REGION              = boto3.Session().region_name
SM_CLIENT           = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT           = boto3.client("s3", region_name=REGION)

MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found — using local file fallback (EFS)")
mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)  # sagemaker-mlflow resolves arn: scheme

EXPERIMENT_NAME     = "chicago-ps3-root-cause"
MLFLOW_MODEL_NAME   = "ps3-root-cause-classification"
MODEL_REGISTRY_NAME = "chicago-ps3-root-cause"
FS_GROUP_NAME       = "chicago-ps3-features"
ENDPOINT_NAME       = "chicago-ps3-rootcause-v1"

# Primary target: 3-class failure level (available via CTA mirror)
# Original target (9-class fault codes) BLOCKED — SVN_STAGE.U_FS_FAULT_CODES = 0 rows
# AE_FAILURE_LEVEL Ventra taxonomy confirmed 2026-06-23 (feasibility updated 55% → 70%)
TARGET          = "failure_level_label"    # MINOR=0 / MAJOR=1 / CRITICAL=2
TARGET_NUMERIC  = "failure_level"          # 1/2/3 → encoded to 0/1/2
NUM_CLASSES     = 3
CLASS_NAMES     = ["MINOR","MAJOR","CRITICAL"]

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Gold S3: {GOLD_S3}")
print("NOTE: PS3 feasibility=70% (updated 2026-06-23). SVN_STAGE.U_FS_FAULT_CODES/ACTION_CODES=0 rows.")
print("      9-class fc_description BLOCKED. Using 3-class failure_level_label.")

In [ ]:
# ── CELL 3b : Checkpoint — load saved state (skip Cells 4-7 if found) ────────
# Saves: df, train/val/test_df, STRUCT_COLS, TARGET_COL, ID_COLS, text_cols
# Path : /home/sagemaker-user/PS3/checkpoints/  (EFS — survives instance stop)
# Usage: Run Cells 1-3-3b → if CHECKPOINT_LOADED=True → jump to Cell 9 (Helpers)
#        Set FORCE_RELOAD=True to re-pull from S3 and rebuild features.

import os, joblib, pandas as _pd_ckpt

CKPT_DIR      = "/home/sagemaker-user/PS3/checkpoints"
FORCE_RELOAD  = False
CHECKPOINT_LOADED = False

_ckpt_df   = os.path.join(CKPT_DIR, "df.parquet")
_ckpt_meta = os.path.join(CKPT_DIR, "meta.joblib")

if not FORCE_RELOAD and os.path.exists(_ckpt_df) and os.path.exists(_ckpt_meta):
    print("[CHECKPOINT] Loading from", CKPT_DIR)
    df       = _pd_ckpt.read_parquet(_ckpt_df)
    train_df = _pd_ckpt.read_parquet(os.path.join(CKPT_DIR, "train_df.parquet"))
    val_df   = _pd_ckpt.read_parquet(os.path.join(CKPT_DIR, "val_df.parquet"))
    test_df  = _pd_ckpt.read_parquet(os.path.join(CKPT_DIR, "test_df.parquet"))
    _meta         = joblib.load(_ckpt_meta)
    STRUCT_COLS   = _meta["STRUCT_COLS"]
    TARGET_COL    = _meta["TARGET_COL"]
    ID_COLS       = _meta["ID_COLS"]
    text_cols     = _meta["text_cols"]
    CHECKPOINT_LOADED = True
    print(f"  df           : {df.shape}")
    print(f"  train / val / test : {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
    print(f"  STRUCT_COLS  : {len(STRUCT_COLS)}  |  TARGET_COL: {TARGET_COL}")
    print(f"  text_cols    : {text_cols}")
    print()
    print("  CHECKPOINT LOADED — skip Cells 4, 5, 6, 7 and jump to Cell 8 (Feature Store)")
    print("  or Cell 9 (Helpers) if Feature Store is not needed.")
else:
    print("[CHECKPOINT] Not found or FORCE_RELOAD=True — running full pipeline (Cells 4-7).")
    print("  Checkpoint will be saved automatically at end of Cell 7.")


In [ ]:
# ── CELL 4 : Load Gold table (Parquet on S3) ──────────────────────────────────
print(f"Loading Gold table from: {GOLD_S3}")
df = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
# Gold SQL uses lowercase "device_id"; normalize to DEVICE_ID used throughout notebook
df = df.rename(columns={c: c.upper() for c in ["device_id", "availability_event_id"]
                        if c in df.columns and c.upper() not in df.columns})
GOLD_VERSION = None   # Parquet has no version; use None for MLflow tags
DATA_SOURCE  = "parquet"

df["transit_day"] = pd.to_datetime(df["transit_day"])

# Encode failure_level 1/2/3 → 0/1/2 (MINOR/MAJOR/CRITICAL)
if TARGET_NUMERIC in df.columns:
    df[TARGET] = df[TARGET_NUMERIC].fillna(1).astype(int) - 1
    df[TARGET] = df[TARGET].clip(0, NUM_CLASSES - 1)
    # Remap to contiguous 0-based if not all failure levels present in data
    _present_idx = sorted(df[TARGET].dropna().astype(int).unique())
    if _present_idx != list(range(len(_present_idx))):
        _remap = {old: new for new, old in enumerate(_present_idx)}
        df[TARGET] = df[TARGET].map(_remap)
        NUM_CLASSES = len(_present_idx)
        CLASS_NAMES = [CLASS_NAMES[i] for i in _present_idx]
        print(f"Note: {len(_present_idx)}/{NUM_CLASSES} failure levels present. "
              f"Remapped {_remap} → CLASS_NAMES={CLASS_NAMES}")

print(f"Parquet source  : confirmed")
print(f"Shape           : {df.shape}")
print(f"Date range      : {df['transit_day'].min().date()}  →  {df['transit_day'].max().date()}")
print(f"\nTarget distribution (0=MINOR, 1=MAJOR, 2=CRITICAL):\n{df[TARGET].value_counts()}")
print(f"\nData source: {DATA_SOURCE}  |  Gold version: {GOLD_VERSION}")

In [ ]:
# ── CELL 5 : Rule-based root_cause_category (8-class) ───────────────────────
# Derived from NLP text match on AE_FAULT_DESCRIPTION / AE_SYMPTOM.
# No ML model needed — deterministic rule engine.
# This is the workaround for SVN_STAGE.U_FS_ACTION_CODES = 0 rows.

CATEGORIES = [
    ("CASH_HANDLING", ["cash","note","coin","jam","dispense","banknote","bna","ctu"]),
    ("PRINTER",       ["print","paper","receipt","roll","thermal"]),
    ("CARD_READER",   ["card","chip","swipe","reader","csc","nfc","contactless"]),
    ("GATE_MECH",     ["gate","arm","barrier","motor","turnstile"]),
    ("COMMS",         ["comms","network","timeout","disconnect","connect","wifi","tcp"]),
    ("POWER",         ["power","voltage","ups","battery","surge","trip"]),
    ("SOFTWARE",      ["software","firmware","crash","reboot","update","exception"]),
    ("UNKNOWN",       []),   # catch-all
]

def classify_root_cause(text):
    if pd.isna(text):
        return "UNKNOWN"
    t = str(text).lower()
    for cat, keywords in CATEGORIES[:-1]:
        if any(k in t for k in keywords):
            return cat
    return "UNKNOWN"

text_cols = [c for c in ["AE_FAULT_DESCRIPTION","AE_SYMPTOM","AE_PROBLEM"]
             if c in df.columns]
if text_cols:
    combined_text = df[text_cols].fillna("").agg(" ".join, axis=1)
    df["root_cause_category"] = combined_text.apply(classify_root_cause)
    print(f"\nRoot cause distribution:\n{df['root_cause_category'].value_counts()}")

    # Log rule stats to MLflow
    with mlflow.start_run(run_name="rule_based_root_cause") as run:
        mlflow.set_tags({"ps":"PS3","model_type":"rule_based","city":"chicago"})
        dist = df["root_cause_category"].value_counts()
        for cat, count in dist.items():
            mlflow.log_metric(f"cat_{cat}", int(count))
        mlflow.log_param("n_categories",len(CATEGORIES))
        mlflow.log_param("n_text_cols",len(text_cols))
        mlflow.log_param("coverage",
            round((df["root_cause_category"]!="UNKNOWN").mean(),4))
        # Save rule config
        rule_config = {c:kw for c,kw in CATEGORIES}
        with open("/tmp/root_cause_rules.json","w") as f:
            json.dump(rule_config,f,indent=2)
        mlflow.log_artifact("/tmp/root_cause_rules.json","rules")
        RULES_RUN = run.info.run_id
    print(f"\nRule-based categories logged. Coverage: {(df['root_cause_category']!='UNKNOWN').mean():.1%}")
else:
    print("No text columns in Gold — rule-based root cause skipped")

In [ ]:
# ── CELL 6 : EDA ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

pd.Series(CLASS_NAMES)[df[TARGET].value_counts().sort_index().index].pipe(
    lambda x: df[TARGET].value_counts().sort_index()).plot(
    kind="bar", ax=axes[0], color=["#43A047","#F57F17","#C62828"], edgecolor="white")
_present = df[TARGET].value_counts().sort_index().index.tolist()
axes[0].set_xticklabels(
    [CLASS_NAMES[i] if i < len(CLASS_NAMES) else str(i) for i in _present],
    rotation=0)
axes[0].set_title("Failure Level Distribution")

if text_cols:
    df["root_cause_category"].value_counts().plot(kind="barh",ax=axes[1],color="#1565C0")
    axes[1].set_title("Root Cause Category (rule-based)")

df.groupby("transit_day")[TARGET].value_counts().unstack(fill_value=0).plot(
    ax=axes[2], linewidth=0.7)
axes[2].set_title("Failure level over time")

plt.tight_layout(); plt.savefig("/tmp/ps3_eda.png",dpi=100); plt.show()

In [ ]:
# ── CELL 7 : Feature engineering ─────────────────────────────────────────────
TARGET_COL  = TARGET
# Leakage guard:
# failure_level / is_chargeable / effective_severity / severity all directly
# encode the target class — must be excluded from STRUCT_COLS.
_EXPLICIT_LEAK = [
    "DEVICE_ID","transit_day","split",
    "AE_FAULT_DESCRIPTION","AE_SYMPTOM","AE_PROBLEM","AE_RESOLUTION",
    "root_cause_category","failure_level","failure_level_label",
    "is_chargeable","is_device_fault","is_hardware_oos_event",
    "failure_level_desc","failure_level_label_enc",
    "effective_severity","effective_severity_num",
    "severity","severity_num","severity_code",
    "is_oos","is_oos_event","oos_flag","is_out_of_service",
    "failure_count","failures_count","total_failures",
    "max_failure_level","min_failure_level",
    "total_failure_min","avg_failure_min",
]
ID_COLS = [c for c in _EXPLICIT_LEAK if c in df.columns]
# Also auto-exclude any column whose name contains these keywords (belt-and-suspenders)
_LEAK_KEYWORDS = ("failure_level","is_chargeable","is_device_fault",
                  "effective_severity","is_hardware_oos","is_oos")
_auto_leak = [c for c in df.columns
              if any(kw in c.lower() for kw in _LEAK_KEYWORDS)
              and c not in ID_COLS]
ID_COLS = list(dict.fromkeys(ID_COLS + _auto_leak))
print(f"Leakage columns excluded ({len(ID_COLS)-5}): ")
print([c for c in ID_COLS if c not in ["DEVICE_ID","transit_day","split","AE_FAULT_DESCRIPTION","AE_SYMPTOM"]])

# Structured numeric features
STRUCT_COLS = [
    c for c in df.columns
    if c not in ID_COLS + [TARGET_COL]
    and pd.api.types.is_numeric_dtype(df[c])
]
df[STRUCT_COLS] = df[STRUCT_COLS].fillna(df[STRUCT_COLS].median())

# Text features: combine available NLP columns
if text_cols:
    df["combined_text"] = df[text_cols].fillna("").agg(" ".join, axis=1)
    df["combined_text"] = df["combined_text"].str.lower().str.replace(
        r"[^a-z\s]","",regex=True)

print(f"Structured features : {len(STRUCT_COLS)}")
print(f"Text features       : {len(text_cols)} columns → combined_text")

# Temporal splits (by transit_day)
train_df = df[df["transit_day"] <  "2025-10-01"]
val_df   = df[(df["transit_day"] >= "2025-10-01") & (df["transit_day"] < "2026-01-01")]
test_df  = df[df["transit_day"] >= "2026-01-01"]

# Debug: show device type distribution across splits
_dc = next((c for c in ["mars_device_category","device_type","device_category"] if c in df.columns), None)
if _dc:
    print(f"Device column: {_dc}")
    print("  Train:", dict(train_df[_dc].value_counts()))
    print("  Val:  ", dict(val_df[_dc].value_counts()))
    print("  Test: ", dict(test_df[_dc].value_counts()))
else:
    print(f"[WARN] No device column found. Available: {[c for c in df.columns if 'device' in c.lower() or 'cat' in c.lower()][:10]}")
X_train_s = train_df[STRUCT_COLS].values; y_train = train_df[TARGET_COL].values
X_val_s   = val_df[STRUCT_COLS].values;   y_val   = val_df[TARGET_COL].values
X_test_s  = test_df[STRUCT_COLS].values;  y_test  = test_df[TARGET_COL].values
X_cv      = np.vstack([X_train_s,X_val_s])
y_cv      = np.concatenate([y_train,y_val])

print(f"Train:{len(X_train_s):,}  Val:{len(X_val_s):,}  Test:{len(X_test_s):,}")

# ── Save checkpoint after feature engineering ────────────────────────────
import os, joblib as _jl
_ckpt_dir = "/home/sagemaker-user/PS3/checkpoints"
os.makedirs(_ckpt_dir, exist_ok=True)
df.to_parquet(       os.path.join(_ckpt_dir, "df.parquet"),       index=True)
train_df.to_parquet( os.path.join(_ckpt_dir, "train_df.parquet"), index=True)
val_df.to_parquet(   os.path.join(_ckpt_dir, "val_df.parquet"),   index=True)
test_df.to_parquet(  os.path.join(_ckpt_dir, "test_df.parquet"),  index=True)
_jl.dump({
    "STRUCT_COLS": STRUCT_COLS,
    "TARGET_COL":  TARGET_COL,
    "ID_COLS":     ID_COLS,
    "text_cols":   text_cols if "text_cols" in dir() else [],
}, os.path.join(_ckpt_dir, "meta.joblib"))
print(f"[CHECKPOINT] Saved to {_ckpt_dir}")
print(f"  df={df.shape}  train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
print(f"  Next kernel restart: run Cells 1-3-3b only, then jump to Cell 9.")


In [ ]:
# ── CELL 8 : Feature Store ────────────────────────────────────────────────────
fs_df = train_df[STRUCT_COLS+[TARGET_COL,"DEVICE_ID","transit_day"]].copy()
fs_df["event_time"] = fs_df["transit_day"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
fs_df["DEVICE_ID"]  = fs_df["DEVICE_ID"].astype(str)
fs_df               = fs_df.drop(columns=["transit_day"])

def dtype_to_fs(dtype):
    if pd.api.types.is_float_dtype(dtype):   return FeatureTypeEnum.FRACTIONAL
    if pd.api.types.is_integer_dtype(dtype): return FeatureTypeEnum.INTEGRAL
    return FeatureTypeEnum.STRING

# Delete-and-recreate so schema changes (e.g. after leakage fix) always land cleanly
_sm_client_fg = SM_SESSION.boto_session.client("sagemaker", region_name=REGION)
try:
    _sm_client_fg.delete_feature_group(FeatureGroupName=FS_GROUP_NAME)
    print(f"Deleted existing Feature Group: {FS_GROUP_NAME} — waiting 60s ...")
    time.sleep(60)
except _sm_client_fg.exceptions.ResourceNotFound:
    pass
except Exception as _del_e:
    print(f"Delete attempt: {_del_e}")

feature_group = FeatureGroup(name=FS_GROUP_NAME, sagemaker_session=SM_SESSION,
    feature_definitions=[
        FeatureDefinition(feature_name=c, feature_type=dtype_to_fs(fs_df[c].dtype))
        for c in fs_df.columns])
try:
    feature_group.create(
        s3_uri=f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}",
        record_identifier_name="DEVICE_ID", event_time_feature_name="event_time",
        role_arn=ROLE, enable_online_store=True,
        description="PS3 root cause classification features — Chicago Ventra")
    status = ""
    while status != "Created":
        time.sleep(15)
        status = feature_group.describe().get("FeatureGroupStatus")
        print(f"  {status}")
    print(f"Feature Group created: {FS_GROUP_NAME}")
except Exception as e:
    print(f"Feature Group create: {e}")

feature_group.ingest(data_frame=fs_df.reset_index(drop=True),max_workers=4,wait=True)
print(f"Ingested {len(fs_df):,} → {FS_GROUP_NAME}")

In [ ]:
# ── CELL 9 : Helpers ──────────────────────────────────────────────────────────
COMMON_TAGS = {"ps":"PS3","target":"failure_level_label (3-class)","city":"chicago",
               "gold_version":str(GOLD_VERSION),"feasibility":"70pct",
               "architecture":"CUBIC-MARS-AWS-v6"}

def evaluate_mc(model, X, y, split="test"):
    """AUC-macro, F1-macro, F1-weighted for one split."""
    prob = model.predict_proba(X)
    pred = prob.argmax(axis=1)
    _n_cls = len(set(y.tolist() if hasattr(y,"tolist") else list(y)))
    if _n_cls < 2:
        auc = -1.0  # MLflow cannot store NaN; -1.0 = "undefined (single class)"
        print(f"  [warn] {split}: only {_n_cls} class in y — AUC logged as -1")
    else:
        try:   auc = roc_auc_score(y, prob, average="macro", multi_class="ovr")
        except Exception as _e: auc = -1.0; print(f"  [warn] AUC: {_e}")
    return {
        f"{split}_auc_macro"  : round(auc, 4),
        f"{split}_f1_macro"   : round(f1_score(y, pred, average="macro",    zero_division=0), 4),
        f"{split}_f1_weighted": round(f1_score(y, pred, average="weighted", zero_division=0), 4),
        f"{split}_accuracy"   : round((pred == y).mean(), 4),
    }

def evaluate_all_splits(model, X_tr, y_tr, X_va, y_va, X_te, y_te):
    """Return train+val+test metrics dict in one call — easy to log_metrics."""
    return {**evaluate_mc(model, X_tr, y_tr, "train"),
            **evaluate_mc(model, X_va, y_va, "val"),
            **evaluate_mc(model, X_te, y_te, "test")}

def log_cm_mc(model, X, y, name):
    """Log confusion matrix PNG to MLflow."""
    pred = model.predict_proba(X).argmax(axis=1)
    cm   = confusion_matrix(y, pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"{name} — Confusion Matrix (test)")
    fig.tight_layout()
    fig.savefig(f"/tmp/cm_{name}.png", dpi=80)
    plt.show(); plt.close(fig)
    mlflow.log_artifact(f"/tmp/cm_{name}.png")

def log_roc_curves(model, X, y, name):
    """OVR ROC curves. Uses manual binarization so shape is always (n, NUM_CLASSES)."""
    from sklearn.metrics import roc_curve, auc as sk_auc
    prob            = model.predict_proba(X)
    classes_present = set(np.unique(y).tolist())
    # Manual binarize: always (n, NUM_CLASSES) regardless of classes in y
    y_bin = np.zeros((len(y), NUM_CLASSES), dtype=int)
    for _k in range(NUM_CLASSES):
        y_bin[:, _k] = (np.asarray(y) == _k).astype(int)
    colors = ["#43A047", "#F57F17", "#C62828"]
    fig, ax = plt.subplots(figsize=(7, 5))
    for k, (cls, col) in enumerate(zip(CLASS_NAMES, colors)):
        if k not in classes_present:
            ax.plot([], [], color=col, lw=1.8,
                    label=f"{cls}  (absent in split)", ls="--")
            continue
        if y_bin[:, k].sum() == 0:
            continue  # no positive samples for this class
        try:
            fpr, tpr, _ = roc_curve(y_bin[:, k], prob[:, k])
            auc_k = sk_auc(fpr, tpr)
            ax.plot(fpr, tpr, color=col, lw=1.8,
                    label=f"{cls}  AUC={auc_k:.3f}")
        except Exception as _re:
            print(f"  {cls} ROC skipped: {_re}")
    ax.plot([0,1],[0,1],"k--",lw=0.8,label="random")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    _absent = [CLASS_NAMES[k] for k in range(NUM_CLASSES) if k not in classes_present]
    _note = f" [absent: {chr(44).join(_absent)}]" if _absent else ""
    ax.set_title(f"{name} -- ROC Curves (OVR, test set){_note}")
    ax.legend(fontsize=9); fig.tight_layout()
    fig.savefig(f"/tmp/roc_{name}.png", dpi=100)
    plt.show(); plt.close(fig)
    mlflow.log_artifact(f"/tmp/roc_{name}.png")

def log_feature_importance(model, feature_names, name, top_n=25):
    """Bar chart of feature importance → MLflow. Works for XGB, LGB, CatBoost."""
    try:
        if hasattr(model, "feature_importances_"):
            imp = model.feature_importances_
        else:
            return
        s = pd.Series(imp, index=feature_names[:len(imp)]).nlargest(top_n)
        fig, ax = plt.subplots(figsize=(8, max(4, top_n // 3)))
        s[::-1].plot(kind="barh", ax=ax, color="steelblue")
        ax.set_xlabel("Importance")
        ax.set_title(f"{name} — Feature Importance (top {top_n})")
        fig.tight_layout()
        fig.savefig(f"/tmp/fi_{name}.png", dpi=100)
        plt.show(); plt.close(fig)
        mlflow.log_artifact(f"/tmp/fi_{name}.png")
    except Exception as _e:
        print(f"  [WARN] feature importance skipped: {_e}")

def print_split_summary(name, metrics):
    """Pretty-print train/val/test metrics side by side."""
    print(f"\n{'─'*60}")
    print(f"  {name}")
    print(f"  {'Split':<8} {'AUC-macro':>10} {'F1-macro':>10} {'Accuracy':>10}")
    for sp in ["train","val","test"]:
        auc_  = metrics.get(f"{sp}_auc_macro",  "—")
        f1_   = metrics.get(f"{sp}_f1_macro",   "—")
        acc_  = metrics.get(f"{sp}_accuracy",    "—")
        print(f"  {sp:<8} {str(auc_):>10} {str(f1_):>10} {str(acc_):>10}")
    print(f"{'─'*60}")

def log_per_device_metrics(model, X_tr, y_tr, X_va, y_va, X_te, y_te,
                            train_df, val_df, test_df, model_name,
                            feature_type="structured"):
    """
    Print and log per-device AUC/F1/Acc for TVM, GATE, VALIDATOR across
    train / val / test.  Works for structured (array) and text (Series) inputs.
    feature_type: "structured" (X is numpy array) or "text" (X is pd.Series).
    """
    _dev_col = next((c for c in ["mars_device_category","device_type","device_category"]
                     if c in train_df.columns), None)
    if _dev_col is None:
        return

    _EXPECTED = ["TVM", "GATE", "VALIDATOR"]
    _splits = [("train", train_df, X_tr, y_tr, "< 2025-10-01"),
               ("val",   val_df,   X_va, y_va, "Oct-Dec 2025"),
               ("test",  test_df,  X_te, y_te, ">= 2026-01-01")]

    print(f"\n  [{model_name}] Per-device breakdown:")
    _h = ["Device","Split","N","AUC","F1-mac","Acc"]
    print(f"  {'':2}{_h[0]:<12}{_h[1]:<8}{_h[2]:>6}  {_h[3]:>6}  {_h[4]:>7}  {_h[5]:>7}")
    print("  " + "-"*55)

    _rows = []
    for split_name, df_s, X_s, y_s, period in _splits:
        _cats_here = set(df_s[_dev_col].dropna().unique())
        for cat in _EXPECTED:
            if cat not in _cats_here:
                print(f"  {cat:<12}{split_name:<8}{'--':>6}  ABSENT")
                continue
            mask = df_s[_dev_col].values == cat
            if mask.sum() < 5:
                continue
            if feature_type == "text":
                X_c = X_s[mask]
            else:
                X_c = X_s[mask]
            y_c  = y_s[mask]
            prob = model.predict_proba(X_c)
            pred = prob.argmax(axis=1)
            acc  = float((pred == y_c).mean())
            f1m  = float(f1_score(y_c, pred, average="macro", zero_division=0))
            if len(np.unique(y_c)) >= 2:
                try:   auc_v = float(roc_auc_score(y_c, prob, multi_class="ovr", average="macro"))
                except: auc_v = -1.0
            else:
                auc_v = -1.0  # single class, AUC undefined
            _auc_s = f"{auc_v:.3f}" if auc_v == auc_v else "  n/a"
            print(f"  {cat:<12}{split_name:<8}{mask.sum():>6}  {_auc_s:>6}  {f1m:>7.3f}  {acc:>7.3f}")
            _rows.append({"device": cat, "split": split_name,
                          "n": int(mask.sum()), "auc": auc_v,
                          "f1_macro": f1m, "accuracy": acc})

    # Log to active MLflow run
    try:
        for row in _rows:
            if row["n"] == 0 or row["auc"] != row["auc"]:
                continue
            pfx = f"{row['device'].lower()}_{row['split']}"
            mlflow.log_metrics({
                f"{pfx}_auc":      round(row["auc"], 4),
                f"{pfx}_f1_macro": round(row["f1_macro"], 4),
                f"{pfx}_accuracy": round(row["accuracy"], 4),
            })
    except Exception:
        pass  # no active run or mlflow not started yet


In [ ]:
# ── CELL 10 : MODEL 1 — NLP Pipeline (TF-IDF + Logistic Regression) ──────────
# Structured NLP baseline — uses only text features.
# Establishes how much information is in fault description text alone.

if text_cols and "combined_text" in df.columns:
    with mlflow.start_run(run_name="tfidf_logistic") as run:
        mlflow.set_tags({**COMMON_TAGS,"model_type":"nlp_tfidf_lr"})

        nlp_pipe = Pipeline([
            ("tfidf", TfidfVectorizer(
                max_features  = 5000,
                ngram_range   = (1, 2),
                min_df        = 3,
                strip_accents = "unicode",
                sublinear_tf  = True,
            )),
            ("clf", LogisticRegression(
                C             = 1.0,
                class_weight  = "balanced",
                max_iter      = 1000,
                random_state  = 42,
            )),
        ])

        nlp_pipe.fit(train_df["combined_text"], y_train)

        # Wrap for predict_proba interface consistency
        class NLPWrapper:
            def __init__(self, pipe): self.pipe = pipe
            def predict_proba(self, X): return self.pipe.predict_proba(X)
            def predict(self, X): return self.pipe.predict(X)

        vm = evaluate_mc(nlp_pipe, val_df["combined_text"],  y_val,  "val")
        tm = evaluate_mc(nlp_pipe, test_df["combined_text"], y_test, "test")
        mlflow.log_params({"max_features":5000,"ngram_range":"(1,2)","C":1.0})
        mlflow.log_metrics({k:(v if v==v else -1.0) for k,v in {**vm,**tm}.items()})
        log_cm_mc(nlp_pipe, test_df["combined_text"], y_test, "tfidf_lr")
        log_per_device_metrics(
            nlp_pipe,
            train_df["combined_text"].values, y_train,
            val_df["combined_text"].values,   y_val,
            test_df["combined_text"].values,  y_test,
            train_df, val_df, test_df, "tfidf_lr", feature_type="text")

        sig = infer_signature(train_df["combined_text"].values[:3],
                               nlp_pipe.predict_proba(train_df["combined_text"].values[:3]))
        try:
            mlflow.sklearn.log_model(nlp_pipe,"model",signature=sig)
        except Exception as _lm_e:
            print(f"  [warn] log_model: {_lm_e}")
        NLP_RUN = run.info.run_id

    print(f"TF-IDF+LR  val_auc={vm['val_auc_macro']}  test_auc={tm['test_auc_macro']}")
else:
    print("No text columns — NLP pipeline skipped"); NLP_RUN = None

In [ ]:
# ── CELL 11 : MODEL 2 — XGBoost multiclass (structured features) ─────────────
with mlflow.start_run(run_name="xgboost_multiclass") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"xgboost_multiclass"})

    xgb = XGBClassifier(
        n_estimators    = 500,
        max_depth       = 6,
        learning_rate   = 0.05,
        subsample       = 0.8,
        colsample_bytree= 0.8,
        objective       = "multi:softprob",
        num_class       = NUM_CLASSES,
        eval_metric     = "mlogloss",
        early_stopping_rounds = 40,
        tree_method     = "hist",
        random_state    = 42,
        n_jobs          = -1,
    )
    xgb.fit(X_train_s, y_train, eval_set=[(X_val_s,y_val)], verbose=100)

    all_m = evaluate_all_splits(xgb, X_train_s, y_train, X_val_s, y_val, X_test_s, y_test)
    mlflow.log_params({"n_estimators":500,"max_depth":6,"num_class":NUM_CLASSES,
                       "best_iter":xgb.best_iteration})
    mlflow.log_metrics(all_m)
    log_cm_mc(xgb, X_test_s, y_test, "xgb_mc")
    log_roc_curves(xgb, X_test_s, y_test, "xgb_mc")
    log_feature_importance(xgb, STRUCT_COLS, "xgb_mc")
    log_per_device_metrics(xgb, X_train_s, y_train, X_val_s, y_val,
                        X_test_s, y_test,
                        train_df, val_df, test_df, "xgb_mc")
    print(classification_report(y_test, xgb.predict_proba(X_test_s).argmax(1),
    target_names=CLASS_NAMES))
    sig = infer_signature(X_train_s, xgb.predict_proba(X_train_s))
    try: mlflow.xgboost.log_model(xgb, "model", signature=sig)
    except Exception as _lm_e: print(f"  [warn] log_model: {_lm_e}")
    XGB_RUN = run.info.run_id
print_split_summary("XGBoost multiclass", all_m)

In [ ]:
# ── CELL 12 : MODEL 3 — LightGBM multiclass ───────────────────────────────────
with mlflow.start_run(run_name="lightgbm_multiclass") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"lightgbm_multiclass"})
    lgb=LGBMClassifier(
        n_estimators=500,num_leaves=63,learning_rate=0.05,
        subsample=0.8,colsample_bytree=0.8,
        objective="multiclass",num_class=NUM_CLASSES,
        class_weight="balanced",n_jobs=-1,random_state=42,verbose=-1)
    lgb.fit(X_train_s,y_train,eval_set=[(X_val_s,y_val)],
            callbacks=[early_stopping(40,verbose=False),log_evaluation(100)])
    all_m = evaluate_all_splits(lgb, X_train_s, y_train, X_val_s, y_val, X_test_s, y_test)
    mlflow.log_params({"n_estimators":500,"num_class":NUM_CLASSES,
                       "best_iter":lgb.best_iteration_})
    mlflow.log_metrics(all_m)
    log_cm_mc(lgb, X_test_s, y_test, "lgb_mc")
    log_roc_curves(lgb, X_test_s, y_test, "lgb_mc")
    log_feature_importance(lgb, STRUCT_COLS, "lgb_mc")
    log_per_device_metrics(lgb, X_train_s, y_train, X_val_s, y_val,
                        X_test_s, y_test,
                        train_df, val_df, test_df, "lgb_mc")
    print(classification_report(y_test, lgb.predict_proba(X_test_s).argmax(1),
    target_names=CLASS_NAMES))
    sig = infer_signature(X_train_s, lgb.predict_proba(X_train_s))
    try:
        mlflow.lightgbm.log_model(lgb, "model", signature=sig)
    except Exception as _lm_e:
        print(f"  [warn] log_model: {_lm_e}")
    LGB_RUN = run.info.run_id
print_split_summary("LightGBM multiclass", all_m)

In [ ]:
# ── CELL 13 : MODEL 4 — CatBoost multiclass ──────────────────────────────────
cat_idx=[i for i,c in enumerate(STRUCT_COLS) if "device_type" in c or "enc" in c]
with mlflow.start_run(run_name="catboost_multiclass") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"catboost_multiclass"})
    _cb_weight_map = {"MINOR": 1.0, "MAJOR": 1.5, "CRITICAL": 2.0}
    _cb_weights = [_cb_weight_map[c] for c in CLASS_NAMES]
    cb=CatBoostClassifier(
        iterations=500,depth=7,learning_rate=0.05,
        loss_function="MultiClass",eval_metric="TotalF1",
        class_weights=_cb_weights,  # upweight CRITICAL; auto-sized to active classes
        early_stopping_rounds=40,random_seed=42,task_type="CPU",verbose=100)
    cb.fit(X_train_s,y_train,eval_set=(X_val_s,y_val),
           cat_features=cat_idx if cat_idx else None)
    all_m = evaluate_all_splits(cb, X_train_s, y_train, X_val_s, y_val, X_test_s, y_test)
    mlflow.log_params({"iterations":500,"class_weights":str(_cb_weights),
                       "best_iter":cb.best_iteration_})
    mlflow.log_metrics(all_m)
    log_cm_mc(cb, X_test_s, y_test, "cb_mc")
    log_roc_curves(cb, X_test_s, y_test, "cb_mc")
    log_feature_importance(cb, STRUCT_COLS, "cb_mc")
    log_per_device_metrics(cb, X_train_s, y_train, X_val_s, y_val,
                        X_test_s, y_test,
                        train_df, val_df, test_df, "cb_mc")
    print(classification_report(y_test, cb.predict_proba(X_test_s).argmax(1),
    target_names=CLASS_NAMES))
    cb.save_model("/tmp/ps3_catboost.cbm")
    mlflow.log_artifact("/tmp/ps3_catboost.cbm", "catboost_native")
    sig = infer_signature(X_train_s, cb.predict_proba(X_train_s))
    try: mlflow.sklearn.log_model(cb, "model", signature=sig, serialization_format="cloudpickle")
    except Exception as _lm_e: print(f"  [warn] log_model: {_lm_e}")
    CB_RUN = run.info.run_id
print_split_summary("CatBoost multiclass", all_m)

In [ ]:
# ── CELL 14 : MODEL 5 — Combined NLP + Structured (late fusion) ───────────────
# Only if text columns are available (PS3 70% feasibility scenario — updated 2026-06-23)
# TF-IDF on text → truncated SVD → concatenate with structured features → XGBoost

if text_cols and "combined_text" in df.columns:
    from sklearn.decomposition import TruncatedSVD
    from scipy.sparse import hstack, csr_matrix

    # Stop words include class label names so they cannot leak into SVD features:
    # "critical"/"major"/"minor" are the exact class names in failure_level_label.
    _label_stop_words = [
        "critical","major","minor","warning","severity","level",
        "failure level","fault level","classification","category",
    ]
    tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2),
                            min_df=3, sublinear_tf=True,
                            stop_words=_label_stop_words)
    X_text_train = tfidf.fit_transform(train_df["combined_text"])
    X_text_val   = tfidf.transform(val_df["combined_text"])
    X_text_test  = tfidf.transform(test_df["combined_text"])

    # Reduce text to 50 dense components
    svd = TruncatedSVD(n_components=50, random_state=42)
    X_text_train_d = svd.fit_transform(X_text_train)
    X_text_val_d   = svd.transform(X_text_val)
    X_text_test_d  = svd.transform(X_text_test)

    # ── SVD Semantic Naming ────────────────────────────────────────────
    # Map each SVD component to the closest root cause category using
    # the same keyword lists from Cell 5. Names become txt_cash_handling,
    # txt_card_reader, etc. — readable in SHAP plots and feature importance.
    _vocab = np.array(tfidf.get_feature_names_out())

    # Category keyword map (same as Cell 5 CATEGORIES)
    _CAT_KW = {
        "cash_handling" : ["cash","note","coin","jam","dispense","banknote","bna","ctu","vault","reject"],
        "card_reader"   : ["card","chip","swipe","reader","csc","nfc","contactless","tap","mag","stripe"],
        "printer"       : ["print","paper","receipt","roll","thermal","head","feed","jam","cutter","media"],
        "gate_mech"     : ["gate","arm","barrier","motor","turnstile","sensor","lock","open","close","swing"],
        "comms"         : ["comms","network","timeout","disconnect","connect","wifi","tcp","socket","host","ping"],
        "power"         : ["power","voltage","ups","battery","surge","trip","supply","breaker","current","overload"],
        "software"      : ["software","firmware","crash","reboot","update","exception","error","hang","freeze","boot"],
        "display"       : ["screen","display","lcd","touch","panel","monitor","brightness","pixel","flicker"],
    }

    def _score_component(component_vec, top_n=20):
        """Score a SVD component against each category; return best match."""
        top_words = set(_vocab[component_vec.argsort()[::-1][:top_n]])
        scores = {cat: len(top_words & set(kws)) for cat, kws in _CAT_KW.items()}
        best_cat, best_score = max(scores.items(), key=lambda x: x[1])
        return best_cat if best_score > 0 else "mixed"

    # Build SVD_NAMES with semantic labels; add suffix _2/_3 for duplicates
    SVD_NAMES  = []
    _used_cats = {}   # cat -> count
    _topic_rows = []

    print("\nSVD Semantic Topics (50 components mapped to root-cause categories):")
    _hdr_cmp, _hdr_var = "Component", "Var%"
    print(f"  {_hdr_cmp:<28} {_hdr_var:>6}  Semantic label          Top-8 words")
    print("  " + "-"*100)

    for k in range(svd.n_components):
        top_idx  = svd.components_[k].argsort()[::-1]
        top8     = _vocab[top_idx[:8]]
        var_exp  = svd.explained_variance_ratio_[k]
        cat      = _score_component(svd.components_[k])
        cnt      = _used_cats.get(cat, 0) + 1
        _used_cats[cat] = cnt
        # First occurrence uses plain name; subsequent get numeric suffix
        sem_label = f"txt_{cat}" if cnt == 1 else f"txt_{cat}_{cnt}"
        SVD_NAMES.append(sem_label)
        _top8_str = " | ".join(top8)
        print(f"  svd_{k:<4} {sem_label:<28} ({var_exp:.2%})  {_top8_str}")
        _topic_rows.append({
            "svd_index":           k,
            "semantic_name":       sem_label,
            "category":            cat,
            "variance_explained":  round(float(var_exp), 4),
            "top_10_words":        " | ".join(_vocab[top_idx[:10]]),
        })

    # Summary: how many components per category
    print(f"\n  Category distribution across 50 SVD components:")
    for _c, _n in sorted(_used_cats.items(), key=lambda x: -x[1]):
        print(f"    txt_{_c:<20} {_n:>2} components")
    print(f"  Total variance explained: {svd.explained_variance_ratio_.sum():.1%}")


    # Late fusion: concatenate text components + structured features
    X_fused_train = np.hstack([X_train_s, X_text_train_d])
    X_fused_val   = np.hstack([X_val_s,   X_text_val_d])
    X_fused_test  = np.hstack([X_test_s,  X_text_test_d])
    X_fused_cv    = np.vstack([X_fused_train, X_fused_val])

    with mlflow.start_run(run_name="combined_nlp_structured") as run:
        mlflow.set_tags({**COMMON_TAGS,"model_type":"combined_nlp_structured"})
        mlflow.log_params({"text_components":50,"struct_features":len(STRUCT_COLS),
                           "total_features":X_fused_train.shape[1]})

        xgb_fused = XGBClassifier(
            n_estimators=400, max_depth=6, learning_rate=0.05,
            objective="multi:softprob", num_class=NUM_CLASSES,
            eval_metric="mlogloss", early_stopping_rounds=30,
            tree_method="hist", random_state=42, n_jobs=-1)
        xgb_fused.fit(X_fused_train,y_train,
                       eval_set=[(X_fused_val,y_val)],verbose=100)

        vm=evaluate_mc(xgb_fused,X_fused_val,y_val,"val")
        tm=evaluate_mc(xgb_fused,X_fused_test,y_test,"test")
        all_m = evaluate_all_splits(xgb_fused,
            X_fused_train, y_train, X_fused_val, y_val, X_fused_test, y_test)
        mlflow.log_metrics(all_m)
        log_cm_mc(xgb_fused, X_fused_test, y_test, "combined_nlp")
        log_roc_curves(xgb_fused, X_fused_test, y_test, "combined_nlp")
        # Use descriptive SVD names (built earlier in this cell)
        _fn = SVD_NAMES[:50] if len(SVD_NAMES) >= 50 else SVD_NAMES
        log_feature_importance(xgb_fused, STRUCT_COLS + _fn, "combined_nlp")

        sig=infer_signature(X_fused_train,xgb_fused.predict_proba(X_fused_train))
        try: mlflow.xgboost.log_model(xgb_fused,"model",signature=sig)
        except Exception as _lm_e: print(f"  [warn] log_model: {_lm_e}")
        COMBINED_RUN=run.info.run_id

    print(f"Combined NLP+Struct  val_auc={vm['val_auc_macro']}  test_auc={tm['test_auc_macro']}")
    BEST_MODEL      = xgb_fused
    BEST_FEATURE_SET= "fused"
else:
    BEST_MODEL      = cb
    BEST_FEATURE_SET= "structured"
    COMBINED_RUN    = None

In [ ]:
# ── CELL 15 : Optuna — tune XGBoost multiclass ──────────────────────────────────────────
# SPEED FIX: best params saved to checkpoints/optuna_params.joblib after first
# run. Subsequent runs load params and skip study.optimize() entirely (~0 sec).
# Set FORCE_OPTUNA=True to re-run the search (e.g. after major data changes).
# Trials: 30 (unchanged). CV splits: 5. Timeout: none.

import os as _os, joblib as _jl_opt

_OPT_CKPT    = "/home/sagemaker-user/PS3/checkpoints/optuna_params.joblib"
FORCE_OPTUNA = False
_saved_params = {}
if not FORCE_OPTUNA and _os.path.exists(_OPT_CKPT):
    _saved_params = _jl_opt.load(_OPT_CKPT)
    print(f"[OPTUNA] Loaded saved params: {list(_saved_params.keys())}")
    print("  Set FORCE_OPTUNA=True to re-run search.")

X_opt = X_fused_cv if BEST_FEATURE_SET=="fused" else X_cv
TS_CV = TimeSeriesSplit(n_splits=5)
_study = None  # only set if search runs (used for plots)

_OPT_KEY = "xgb_mc"
if _OPT_KEY in _saved_params and not FORCE_OPTUNA:
    BEST_PARAMS = {**_saved_params[_OPT_KEY]["best_params"],
                   "objective": "multi:softprob", "num_class": NUM_CLASSES,
                   "tree_method": "hist", "random_state": 42, "n_jobs": -1}
    _cv_auc = _saved_params[_OPT_KEY]["best_cv_auc"]
    print(f"[OPTUNA] Params loaded from checkpoint (CV_AUC={_cv_auc:.4f}) ? search skipped")
else:
    print("[OPTUNA] Running Optuna (30 trials, 5 folds) ...")
    def optuna_obj(trial):
        p=dict(
            n_estimators    =trial.suggest_int("n_estimators",200,600),
            max_depth       =trial.suggest_int("max_depth",4,8),
            learning_rate   =trial.suggest_float("learning_rate",0.01,0.15,log=True),
            subsample       =trial.suggest_float("subsample",0.6,1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree",0.6,1.0),
            objective="multi:softprob", num_class=NUM_CLASSES,
            tree_method="hist", random_state=42, n_jobs=-1,
        )
        scores=[]
        for step,(tr,va) in enumerate(TS_CV.split(X_opt)):
            m=XGBClassifier(**p)
            m.fit(X_opt[tr],y_cv[tr],verbose=False)
            try:
                s=roc_auc_score(y_cv[va],m.predict_proba(X_opt[va]),average="macro")
            except: s=0.5
            scores.append(s)
            trial.report(np.mean(scores),step)
            if trial.should_prune(): raise optuna.TrialPruned()
        return np.mean(scores)

    _study=optuna.create_study(direction="maximize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5))
    optuna.logging.set_verbosity(optuna.logging.INFO)
    _study.optimize(optuna_obj,n_trials=30,show_progress_bar=False)
    _cv_auc = _study.best_value
    BEST_PARAMS={**_study.best_params,"objective":"multi:softprob","num_class":NUM_CLASSES,
                 "tree_method":"hist","random_state":42,"n_jobs":-1}
    _saved_params[_OPT_KEY] = {"best_params": _study.best_params, "best_cv_auc": _cv_auc}
    _jl_opt.dump(_saved_params, _OPT_CKPT)
    print(f"[OPTUNA] Params saved ? {_OPT_CKPT}")

with mlflow.start_run(run_name="xgboost_optuna_mc") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"xgboost_optuna_multiclass"})
    mlflow.log_params({**BEST_PARAMS,"optuna_trials":30,"optuna_cv":round(_cv_auc,4)})
    xgb_tuned=XGBClassifier(**BEST_PARAMS)
    X_train_use = X_fused_train if BEST_FEATURE_SET=="fused" else X_train_s
    X_val_use   = X_fused_val   if BEST_FEATURE_SET=="fused" else X_val_s
    X_test_use  = X_fused_test  if BEST_FEATURE_SET=="fused" else X_test_s
    xgb_tuned.fit(X_train_use, y_train, verbose=False)
    vm=evaluate_mc(xgb_tuned,X_val_use,y_val,"val")
    tm=evaluate_mc(xgb_tuned,X_test_use,y_test,"test")
    mlflow.log_metrics({k:(v if v==v else -1.0) for k,v in {**vm,**tm}.items()})
    log_cm_mc(xgb_tuned,X_test_use,y_test,"xgb_optuna_mc")
    log_roc_curves(xgb_tuned, X_test_use, y_test, "xgb_optuna_mc")
    log_per_device_metrics(
        xgb_tuned,
        X_train_use, y_train, X_val_use, y_val, X_test_use, y_test,
        train_df, val_df, test_df, "xgb_optuna")
    if _study is not None:
        for fname, fig_fn in [
            ("optuna_history_ps3.png",      optuna.visualization.matplotlib.plot_optimization_history),
            ("optuna_param_import_ps3.png", optuna.visualization.matplotlib.plot_param_importances)]:
            try:
                fig_fn(_study); plt.tight_layout()
                plt.savefig(f"/tmp/{fname}", dpi=80); plt.close()
                mlflow.log_artifact(f"/tmp/{fname}")
            except Exception: pass
    X_sig = X_test_use[:min(200, X_test_use.shape[0])]
    sig   = infer_signature(X_sig, xgb_tuned.predict_proba(X_sig))
    try: mlflow.xgboost.log_model(xgb_tuned,"model",signature=sig)
    except Exception as _lm_e: print(f"  [warn] log_model: {_lm_e}")
    XGB_TUNED_RUN=run.info.run_id
print(f"XGB(Optuna) mc  val_auc={vm['val_auc_macro']:.4f}  test_auc={tm['test_auc_macro']:.4f}")
print(f"[OPTUNA] Complete. Params at {_OPT_CKPT}")
print("  Next restart: params load automatically ? Optuna skipped.")


In [ ]:
# -- CELL 16 : Per-Device Breakdown for Optuna Champion (train|val|test) ------
# xgb_tuned is trained on ALL device types combined.
# This cell shows TVM / GATE / VALIDATOR metrics for every split so VALIDATOR
# is visible in train and val even though it has no data in test (>= 2026-01-01).

from sklearn.metrics import (f1_score, roc_auc_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve)
from sklearn.metrics import auc as sk_auc
import warnings

_dev_col16 = next((c for c in ["mars_device_category","device_type","device_category"]
                   if c in train_df.columns), None)

if _dev_col16 is None:
    print("[SKIP] No device category column")
else:
    _X_tr16 = X_train_use
    _X_va16 = X_val_use
    _X_te16 = X_test_use
    _SPLITS16 = [
        ("train", train_df, _X_tr16, y_train, "< 2025-10-01"),
        ("val",   val_df,   _X_va16, y_val,   "Oct-Dec 2025"),
        ("test",  test_df,  _X_te16, y_test,  ">= 2026-01-01"),
    ]
    _EXPECTED16 = ["TVM", "GATE", "VALIDATOR"]

    def _binarize16(y_arr):
        out = np.zeros((len(y_arr), NUM_CLASSES), dtype=int)
        for _k in range(NUM_CLASSES):
            out[:, _k] = (np.asarray(y_arr) == _k).astype(int)
        return out

    def _eval16(X_s, y_s, df_s, cat):
        mask = df_s[_dev_col16].values == cat
        n = mask.sum()
        if n < 5:
            return None, n
        X_c = X_s[mask]; y_c = y_s[mask]
        prob = xgb_tuned.predict_proba(X_c)
        pred = prob.argmax(axis=1)
        acc  = float((pred == y_c).mean())
        f1m  = float(f1_score(y_c, pred, average="macro", zero_division=0))
        f1w  = float(f1_score(y_c, pred, average="weighted", zero_division=0))
        auc_v = float("nan")
        if len(np.unique(y_c)) >= 2:
            try: auc_v = float(roc_auc_score(y_c, prob, multi_class="ovr", average="macro"))
            except: pass
        return {"n": int(n), "acc": acc, "auc": auc_v,
                "f1_macro": f1m, "f1_weighted": f1w,
                "y_c": y_c, "pred": pred, "prob": prob}, n

    # -- 1. SUMMARY TABLE -----------------------------------------------------
    print("\n" + "="*90)
    print("  Optuna XGBoost — Per-Device Performance (one model, all devices combined)")
    print("="*90)
    _h = ["Device","Split","Period","N","AUC","F1-mac","F1-wtd","Acc"]
    print(f"  {_h[0]:<12}{_h[1]:<8}{_h[2]:<26}{_h[3]:>6}  {_h[4]:>6}  {_h[5]:>7}  {_h[6]:>7}  {_h[7]:>7}")
    print("  " + "-"*90)

    _rows16 = []
    for split_name, df_s, X_s, y_s, period in _SPLITS16:
        _cats_here = set(df_s[_dev_col16].dropna().unique())
        for cat in _EXPECTED16:
            if cat not in _cats_here:
                print(f"  {cat:<12}{split_name:<8}{period:<26}  -- ABSENT (no data in period) --")
                _rows16.append({"device": cat, "split": split_name, "period": period,
                                "n": 0, "auc": None, "f1_macro": None,
                                "f1_weighted": None, "accuracy": None})
                continue
            r, n = _eval16(X_s, y_s, df_s, cat)
            if r is None:
                print(f"  {cat:<12}{split_name:<8}{period:<26}{n:>6}  (< 5 rows)")
                continue
            _auc_s = f"{r['auc']:.3f}" if r["auc"] == r["auc"] else "  n/a"
            print(f"  {cat:<12}{split_name:<8}{period:<26}{r['n']:>6}  {_auc_s:>6}"
                  f"  {r['f1_macro']:>7.3f}  {r['f1_weighted']:>7.3f}  {r['acc']:>7.3f}")
            _rows16.append({"device": cat, "split": split_name, "period": period,
                            "n": r["n"], "auc": r["auc"],
                            "f1_macro": r["f1_macro"],
                            "f1_weighted": r["f1_weighted"],
                            "accuracy": r["acc"]})

    # -- 2. CONFUSION MATRICES: 3 rows (splits) x 3 cols (devices) -----------
    fig_cm16, axes_cm16 = plt.subplots(3, 3, figsize=(15, 12), squeeze=False)
    for ri, (split_name, df_s, X_s, y_s, period) in enumerate(_SPLITS16):
        _cats_here = set(df_s[_dev_col16].dropna().unique())
        for ci, cat in enumerate(_EXPECTED16):
            ax = axes_cm16[ri][ci]
            ax.set_title(f"{cat} | {split_name}", fontsize=9)
            if cat not in _cats_here:
                ax.text(0.5, 0.5, "ABSENT\n(no data in period)",
                        ha="center", va="center", fontsize=9, color="gray",
                        transform=ax.transAxes)
                ax.axis("off"); continue
            r, n = _eval16(X_s, y_s, df_s, cat)
            if r is None:
                ax.text(0.5, 0.5, f"n={n}\n(too few)", ha="center", va="center",
                        fontsize=9, color="gray", transform=ax.transAxes)
                ax.axis("off"); continue
            _auc_lbl = f"{r['auc']:.3f}" if r["auc"] == r["auc"] else "n/a"
            print(f"\n-- {cat} | {split_name} (n={r['n']}, AUC={_auc_lbl}) --")
            print(classification_report(r["y_c"], r["pred"],
                                        target_names=CLASS_NAMES, zero_division=0))
            cm16 = confusion_matrix(r["y_c"], r["pred"], labels=list(range(NUM_CLASSES)))
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                ConfusionMatrixDisplay(cm16, display_labels=CLASS_NAMES).plot(
                    ax=ax, colorbar=False, cmap="Blues")
            ax.set_title(f"{cat} | {split_name}  n={r['n']}  AUC={_auc_lbl}", fontsize=8)

    plt.suptitle("Optuna XGBoost -- Confusion Matrices: TVM/GATE/VALIDATOR x train|val|test",
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig("/tmp/ps3_optuna_per_device_cm.png", dpi=120, bbox_inches="tight")
    plt.show()

    # -- 3. ROC curves on val (VALIDATOR visible) + test ----------------------
    for split_name, df_s, X_s, y_s in [("val",  val_df,  _X_va16, y_val),
                                         ("test", test_df, _X_te16, y_test)]:
        _cats_here = sorted(df_s[_dev_col16].dropna().unique())
        if not _cats_here: continue
        fig_r16, axes_r16 = plt.subplots(1, len(_cats_here),
                                          figsize=(6*len(_cats_here), 4), squeeze=False)
        colors = ["#43A047","#F57F17","#C62828"]
        for ci, cat in enumerate(_cats_here):
            ax2 = axes_r16[0][ci]
            r, _ = _eval16(X_s, y_s, df_s, cat)
            if r is None:
                ax2.text(0.5,0.5,"too few",ha="center",va="center"); continue
            y_bin = _binarize16(r["y_c"])
            for k, (cls, col) in enumerate(zip(CLASS_NAMES, colors)):
                if y_bin[:, k].sum() == 0:
                    ax2.plot([],[],color=col,lw=1.5,label=f"{cls} (absent)",ls="--"); continue
                try:
                    fpr,tpr,_ = roc_curve(y_bin[:,k], r["prob"][:,k])
                    ax2.plot(fpr,tpr,color=col,lw=1.5,label=f"{cls} AUC={sk_auc(fpr,tpr):.3f}")
                except Exception as _e:
                    print(f"  {cat}/{cls}: {_e}")
            ax2.plot([0,1],[0,1],"k--",lw=0.8)
            ax2.set_title(f"{cat} | {split_name} (n={r['n']})", fontsize=10)
            ax2.set_xlabel("FPR"); ax2.set_ylabel("TPR"); ax2.legend(fontsize=8)
        plt.suptitle(f"Optuna XGBoost -- ROC per Device ({split_name})", fontsize=11)
        plt.tight_layout()
        plt.savefig(f"/tmp/ps3_optuna_roc_{split_name}.png", dpi=100, bbox_inches="tight")
        plt.show()

    # -- 4. Log to MLflow under XGB_TUNED_RUN ---------------------------------
    _summary16 = pd.DataFrame(_rows16)
    with mlflow.start_run(run_id=XGB_TUNED_RUN):
        for row in _rows16:
            if row.get("n",0) == 0 or row.get("auc") is None: continue
            pfx = f"{row['device'].lower()}_{row['split']}"
            _safe_auc = round(float(row["auc"]),4) if row["auc"]==row["auc"] else -1.0
            mlflow.log_metrics({
                f"{pfx}_auc":      _safe_auc,
                f"{pfx}_f1_macro": round(float(row["f1_macro"]),4),
                f"{pfx}_accuracy": round(float(row["accuracy"]),4),
                f"{pfx}_n":        int(row["n"]),
            })
        for _art in [
                "/tmp/ps3_optuna_per_device_cm.png",
                "/tmp/ps3_optuna_roc_val.png",
        ]:
            try: mlflow.log_artifact(_art)
            except: pass
        for _art in [
                "/tmp/ps3_optuna_roc_test.png",
        ]:
            try: mlflow.log_artifact(_art)
            except: pass
    print(f"\nOptuna per-device metrics logged to MLflow run: {XGB_TUNED_RUN}")
    CAT_METRICS = _summary16


In [ ]:
# -- CELL 17 SHAP -- Detailed Analysis with descriptive SVD feature names
# Uses xgb_tuned (Optuna champion). SVD components labeled "svd_k:word1|word2|word3"

# Build feature name list using SVD_NAMES built in Cell 14
if BEST_FEATURE_SET == "fused":
    _n_svd    = X_test_use.shape[1] - len(STRUCT_COLS)
    _svd_lbls = SVD_NAMES[:_n_svd] if "SVD_NAMES" in vars() else [f"svd_{k}" for k in range(_n_svd)]
    FEAT_NAMES = STRUCT_COLS + _svd_lbls
else:
    FEAT_NAMES = STRUCT_COLS

_n_shap   = min(1000, X_test_use.shape[0])
X_shap_df = pd.DataFrame(X_test_use[:_n_shap].astype(np.float64), columns=FEAT_NAMES)

explainer = shap.TreeExplainer(xgb_tuned)
sv        = explainer(X_shap_df)
_sv_vals  = sv.values  # [n, features] or [n, features, n_classes]

with mlflow.start_run(run_name="shap_detailed_analysis") as _shap_run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "shap_analysis"})

    # -- 1. GLOBAL bar chart: mean |SHAP| across all classes -----------
    _mean_abs = (np.abs(_sv_vals).mean(axis=(0,2))
                 if _sv_vals.ndim == 3 else np.abs(_sv_vals).mean(axis=0))
    _feat_imp = pd.Series(_mean_abs, index=FEAT_NAMES).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 7))
    _top25  = _feat_imp.head(25)
    _colors = ["#C62828" if "svd_" in n else "#1565C0" for n in _top25.index[::-1]]
    _top25[::-1].plot(kind="barh", ax=ax, color=_colors)
    ax.set_xlabel("Mean |SHAP value|", fontsize=11)
    ax.set_title("PS3 XGBoost -- Global SHAP Feature Importance"
                 " (red=SVD text topic  blue=structured)", fontsize=11, pad=10)
    fig.tight_layout()
    fig.savefig("/tmp/ps3_shap_global_bar.png", dpi=120)
    plt.show(); plt.close(fig)
    mlflow.log_artifact("/tmp/ps3_shap_global_bar.png")

    # -- 2. BEESWARM: direction + distribution -------------------------
    if _sv_vals.ndim == 3:
        _sv_avg = shap.Explanation(
            values      = _sv_vals.mean(axis=2),
            base_values = sv.base_values[:,0] if sv.base_values.ndim>1 else sv.base_values,
            data        = sv.data,
            feature_names = FEAT_NAMES)
    else:
        _sv_avg = sv
    shap.plots.beeswarm(_sv_avg, max_display=25, show=False)
    plt.title("PS3 -- SHAP Beeswarm (top 25 features, classes averaged)", fontsize=10)
    plt.tight_layout()
    plt.gcf().savefig("/tmp/ps3_shap_beeswarm.png", dpi=120)
    plt.show(); plt.close("all")
    mlflow.log_artifact("/tmp/ps3_shap_beeswarm.png")

    # -- 3. PER-CLASS: bar + beeswarm for MINOR / MAJOR / CRITICAL -----
    if _sv_vals.ndim == 3:
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            _bv = sv.base_values[:, cls_idx] if sv.base_values.ndim > 1 else sv.base_values
            _sv_cls = shap.Explanation(
                values        = _sv_vals[:, :, cls_idx],
                base_values   = _bv,
                data          = sv.data,
                feature_names = FEAT_NAMES)
            # Bar
            shap.plots.bar(_sv_cls, max_display=20, show=False)
            plt.gca().set_title(f"PS3 SHAP -- {cls_name} class (mean |SHAP|, top 20)")
            plt.tight_layout()
            plt.gcf().savefig(f"/tmp/ps3_shap_bar_{cls_name}.png", dpi=100)
            plt.show(); plt.close("all")
            mlflow.log_artifact(f"/tmp/ps3_shap_bar_{cls_name}.png")
            # Beeswarm
            shap.plots.beeswarm(_sv_cls, max_display=20, show=False)
            plt.title(f"PS3 SHAP Beeswarm -- {cls_name} (top 20)", fontsize=10)
            plt.tight_layout()
            plt.gcf().savefig(f"/tmp/ps3_shap_beeswarm_{cls_name}.png", dpi=100)
            plt.show(); plt.close("all")
            mlflow.log_artifact(f"/tmp/ps3_shap_beeswarm_{cls_name}.png")

    # -- 4. WATERFALL: one sample per class ----------------------------
    if _sv_vals.ndim == 3:
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            _sample_i = _sv_vals[:, :, cls_idx].sum(axis=1).argmax()
            _bv_s = sv.base_values[_sample_i, cls_idx] if sv.base_values.ndim>1 else sv.base_values[_sample_i]
            _sv_one = shap.Explanation(
                values        = _sv_vals[_sample_i, :, cls_idx],
                base_values   = _bv_s,
                data          = sv.data[_sample_i],
                feature_names = FEAT_NAMES)
            shap.plots.waterfall(_sv_one, max_display=15, show=False)
            plt.title(f"PS3 SHAP Waterfall -- predicted {cls_name}", fontsize=10)
            plt.tight_layout()
            plt.gcf().savefig(f"/tmp/ps3_shap_waterfall_{cls_name}.png", dpi=100)
            plt.show(); plt.close("all")
            mlflow.log_artifact(f"/tmp/ps3_shap_waterfall_{cls_name}.png")

    # -- 5. DEPENDENCE plots for top-3 features -------------------------
    _crit_idx = CLASS_NAMES.index("CRITICAL") if "CRITICAL" in CLASS_NAMES else -1
    _dep_vals = _sv_vals[:, :, _crit_idx] if (_sv_vals.ndim==3 and _crit_idx>=0) else _sv_vals
    _cls_lbl  = CLASS_NAMES[_crit_idx] if _crit_idx >= 0 else "model"
    for feat_name in _feat_imp.index[:3].tolist():
        _fi = FEAT_NAMES.index(feat_name)
        fig_d, ax_d = plt.subplots(figsize=(7, 4))
        ax_d.scatter(X_shap_df[feat_name], _dep_vals[:, _fi],
                     alpha=0.3, s=10, c=_dep_vals[:, _fi], cmap="coolwarm")
        ax_d.axhline(0, color="k", lw=0.7, ls="--")
        ax_d.set_xlabel(feat_name, fontsize=10)
        ax_d.set_ylabel(f"SHAP value ({_cls_lbl} class)", fontsize=10)
        ax_d.set_title(f"SHAP Dependence: {feat_name} -> {_cls_lbl}", fontsize=10)
        fig_d.tight_layout()
        _safe = feat_name.replace(":", "_").replace("|","_")[:40]
        fig_d.savefig(f"/tmp/ps3_shap_dep_{_safe}.png", dpi=100)
        plt.show(); plt.close(fig_d)
        mlflow.log_artifact(f"/tmp/ps3_shap_dep_{_safe}.png")

    # -- 6. Feature importance table CSV --------------------------------
    _imp_tbl = _feat_imp.reset_index()
    _imp_tbl.columns = ["feature", "mean_abs_shap"]
    _imp_tbl["is_text_svd"] = _imp_tbl["feature"].str.startswith("svd_")
    _imp_tbl["rank"] = range(1, len(_imp_tbl)+1)
    print("\nTop 20 SHAP features:")
    print(_imp_tbl.head(20).to_string(index=False))
    SHAP_RUN = _shap_run.info.run_id
    print(f"\nSHAP run: {SHAP_RUN}")


In [ ]:
# -- CELL 18 Per-Device-Category Breakdown across ALL splits -----------------
# One model is trained on ALL device types combined.
# Evaluate separately for TVM / GATE / VALIDATOR on train, val, test.
# VALIDATOR has no records in test (transit_day >= 2026-01-01); it appears in
# train and val -- those splits show its real holdout performance.

from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve)
from sklearn.metrics import auc as sk_auc
import warnings

_dev_col = next((c for c in ["mars_device_category","device_type","device_category"]
                 if c in train_df.columns), None)

if _dev_col is None:
    print("[SKIP] No device category column found in train_df")
else:
    _X_tr = X_fused_train if BEST_FEATURE_SET == "fused" else X_train_s
    _X_va = X_fused_val   if BEST_FEATURE_SET == "fused" else X_val_s
    _X_te = X_fused_test  if BEST_FEATURE_SET == "fused" else X_test_s

    _SPLITS = [
        ("train", train_df, _X_tr, y_train, "< 2025-10-01"),
        ("val",   val_df,   _X_va, y_val,   "2025-10-01..2025-12-31"),
        ("test",  test_df,  _X_te, y_test,  ">= 2026-01-01"),
    ]
    _EXPECTED = ["TVM", "GATE", "VALIDATOR"]

    # All device types present anywhere across splits
    _all_cats = sorted(set(
        v for _, df_s, _, _, _ in _SPLITS
        for v in df_s[_dev_col].dropna().unique()
    ))
    print(f"Device types across all splits: {_all_cats}")
    _missing_global = [d for d in _EXPECTED if d not in _all_cats]
    if _missing_global:
        print(f"  [WARN] {_missing_global} absent from ALL splits -- check Gold table filter")

    # Manual binarize: always (n, NUM_CLASSES) regardless of classes in y
    def _binarize(y_arr):
        out = np.zeros((len(y_arr), NUM_CLASSES), dtype=int)
        for _k in range(NUM_CLASSES):
            out[:, _k] = (np.asarray(y_arr) == _k).astype(int)
        return out

    def _eval_device_split(model, X_s, y_s, df_s, cat):
        """Evaluate model on one device category from one split."""
        mask = df_s[_dev_col].values == cat
        n = mask.sum()
        if n < 5:
            return None
        X_c  = X_s[mask]
        y_c  = y_s[mask]
        prob = model.predict_proba(X_c)
        pred = prob.argmax(axis=1)
        acc  = float((pred == y_c).mean())
        f1m  = float(f1_score(y_c, pred, average="macro",    zero_division=0))
        f1w  = float(f1_score(y_c, pred, average="weighted", zero_division=0))
        if len(np.unique(y_c)) >= 2:
            try:   auc_v = float(roc_auc_score(y_c, prob, multi_class="ovr", average="macro"))
            except: auc_v = float("nan")
        else:
            auc_v = float("nan")
        return {"n": int(n), "acc": acc, "auc": auc_v,
                "f1_macro": f1m, "f1_weighted": f1w,
                "y_c": y_c, "pred": pred, "prob": prob}

    # -- 1. SUMMARY TABLE -------------------------------------------------
    _SEP = "  "
    print("\n" + "="*90)
    print("  Per-Device Performance  (model trained on all devices combined)")
    print("="*90)
    _h = ["Device","Split","Period","N","AUC","F1-mac","F1-wtd","Acc"]
    print(f"  {_h[0]:<12} {_h[1]:<7} {_h[2]:<26} {_h[3]:>6}  {_h[4]:>6}  {_h[5]:>7}  {_h[6]:>7}  {_h[7]:>7}")
    print("  " + "-"*90)

    _summary_rows = []
    for split_name, df_s, X_s, y_s, period in _SPLITS:
        _cats_here = set(df_s[_dev_col].dropna().unique())
        for cat in _EXPECTED:
            if cat not in _cats_here:
                print(f"  {cat:<12} {split_name:<7} {period:<26}  -- ABSENT (no data in this period) --")
                _summary_rows.append({"device": cat, "split": split_name, "period": period,
                                      "n": 0, "auc": None, "f1_macro": None,
                                      "f1_weighted": None, "accuracy": None})
                continue
            r = _eval_device_split(BEST_MODEL, X_s, y_s, df_s, cat)
            if r is None:
                print(f"  {cat:<12} {split_name:<7} {period:<26}  < 5 rows -- skipped")
                continue
            _auc_s = f"{r['auc']:.3f}" if r["auc"] == r["auc"] else " n/a"
            print(f"  {cat:<12} {split_name:<7} {period:<26} {r['n']:>6}  {_auc_s:>6}"
                  f"  {r['f1_macro']:>7.3f}  {r['f1_weighted']:>7.3f}  {r['acc']:>7.3f}")
            _summary_rows.append({"device": cat, "split": split_name, "period": period,
                                  "n": r["n"], "auc": r["auc"],
                                  "f1_macro": r["f1_macro"],
                                  "f1_weighted": r["f1_weighted"],
                                  "accuracy": r["acc"]})

    # -- 2. CONFUSION MATRICES: rows=splits, cols=device types ---------------
    fig_cm, axes_cm = plt.subplots(len(_SPLITS), len(_EXPECTED),
                                    figsize=(5*len(_EXPECTED), 4*len(_SPLITS)),
                                    squeeze=False)
    for ri, (split_name, df_s, X_s, y_s, period) in enumerate(_SPLITS):
        _cats_here = set(df_s[_dev_col].dropna().unique())
        for ci, cat in enumerate(_EXPECTED):
            ax = axes_cm[ri][ci]
            if cat not in _cats_here:
                ax.text(0.5, 0.5, "ABSENT\n(no data in period)",
                        ha="center", va="center", fontsize=9, color="gray",
                        transform=ax.transAxes)
                ax.set_title(f"{cat} | {split_name}", fontsize=8)
                ax.axis("off")
                continue
            r = _eval_device_split(BEST_MODEL, X_s, y_s, df_s, cat)
            if r is None:
                ax.text(0.5, 0.5, "< 5 samples", ha="center", va="center",
                        fontsize=9, color="gray", transform=ax.transAxes)
                ax.axis("off")
                continue
            _auc_lbl = f"{r['auc']:.3f}" if r["auc"] == r["auc"] else "n/a"
            print(f"\n-- {cat} | {split_name} (n={r['n']}, AUC={_auc_lbl}) --")
            print(classification_report(r["y_c"], r["pred"],
                                        target_names=CLASS_NAMES, zero_division=0))
            cm = confusion_matrix(r["y_c"], r["pred"], labels=list(range(NUM_CLASSES)))
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
                    ax=ax, colorbar=False, cmap="Blues")
            ax.set_title(f"{cat} | {split_name}  n={r['n']}  AUC={_auc_lbl}", fontsize=8)

    plt.suptitle("PS3 -- Confusion Matrices: TVM / GATE / VALIDATOR x train|val|test",
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig("/tmp/ps3_per_device_all_splits_cm.png", dpi=120, bbox_inches="tight")
    plt.show()

    # -- 3. ROC curves per device -- val (VALIDATOR visible here) + test -----
    for split_name, df_s, X_s, y_s in [("val",  val_df,  _X_va, y_val),
                                         ("test", test_df, _X_te, y_test)]:
        _cats_here = sorted(df_s[_dev_col].dropna().unique())
        if not _cats_here:
            continue
        fig_roc, axes_roc = plt.subplots(1, len(_cats_here),
                                          figsize=(6*len(_cats_here), 4), squeeze=False)
        colors = ["#43A047", "#F57F17", "#C62828"]
        for ci, cat in enumerate(_cats_here):
            ax2 = axes_roc[0][ci]
            r = _eval_device_split(BEST_MODEL, X_s, y_s, df_s, cat)
            if r is None:
                ax2.text(0.5, 0.5, "too few", ha="center", va="center")
                continue
            y_bin_c = _binarize(r["y_c"])
            for k, (cls, col) in enumerate(zip(CLASS_NAMES, colors)):
                if y_bin_c[:, k].sum() == 0:
                    ax2.plot([], [], color=col, lw=1.5, label=f"{cls} (absent)", ls="--")
                    continue
                try:
                    fpr, tpr, _ = roc_curve(y_bin_c[:, k], r["prob"][:, k])
                    ax2.plot(fpr, tpr, color=col, lw=1.5, label=f"{cls} AUC={sk_auc(fpr,tpr):.3f}")
                except Exception as _re:
                    print(f"  {cat}/{cls} ROC: {_re}")
            ax2.plot([0,1],[0,1],"k--",lw=0.8)
            ax2.set_title(f"{cat} | {split_name} (n={r['n']})", fontsize=10)
            ax2.set_xlabel("FPR"); ax2.set_ylabel("TPR"); ax2.legend(fontsize=8)
        plt.suptitle(f"PS3 -- ROC per Device ({split_name} set)", fontsize=11)
        plt.tight_layout()
        plt.savefig(f"/tmp/ps3_per_device_roc_{split_name}.png", dpi=100, bbox_inches="tight")
        plt.show()

    # -- 4. Save CSV + log to MLflow -----------------------------------------
    _summary_df = pd.DataFrame(_summary_rows)

    with mlflow.start_run(run_name="per_device_all_splits") as _dev_run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "per_device_evaluation"})
        for row in _summary_rows:
            if row.get("n", 0) == 0 or row.get("auc") is None:
                continue
            pfx = f"{row['device'].lower()}_{row['split']}"
            _safe_auc = round(float(row["auc"]), 4) if row["auc"] == row["auc"] else -1.0
            mlflow.log_metrics({
                f"{pfx}_auc":      _safe_auc,
                f"{pfx}_f1_macro": round(float(row["f1_macro"]), 4),
                f"{pfx}_accuracy": round(float(row["accuracy"]), 4),
                f"{pfx}_n":        int(row["n"]),
            })
        for _art in [
                "/tmp/ps3_per_device_all_splits_cm.png",
                "/tmp/ps3_per_device_roc_val.png",
        ]:
            try: mlflow.log_artifact(_art)
            except: pass
        for _art in [
                "/tmp/ps3_per_device_roc_test.png",
        ]:
            try: mlflow.log_artifact(_art)
            except: pass
    print(f"\nPer-device (all splits) logged to MLflow: {_dev_run.info.run_id}")
    DEVICE_METRICS = _summary_df


In [ ]:
# ── CELL 19 : Select champion + register ─────────────────────────────────────
all_runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.test_auc_macro DESC"])
_mask = all_runs["tags.mlflow.runName"].str.contains(
    "xgboost|lightgbm|catboost|combined", na=False)
all_runs = all_runs[_mask]
if len(all_runs) == 0:
    all_runs = mlflow.search_runs(
        experiment_names=[EXPERIMENT_NAME],
        order_by=["metrics.test_auc_macro DESC"],
        filter_string="metrics.test_auc_macro > 0")
if len(all_runs) == 0:
    print("[WARN] No qualifying MLflow runs — skipping champion selection")
    CHAMPION_RUN_ID = None; CHAMPION_NAME = "none"; CHAMPION_AUC = -1.0; MLFLOW_VERSION = 0
else:
    CHAMPION_RUN_ID = all_runs.iloc[0]["run_id"]
    CHAMPION_NAME   = all_runs.iloc[0]["tags.mlflow.runName"]
    CHAMPION_AUC    = all_runs.iloc[0]["metrics.test_auc_macro"]
    print(f"Champion: {CHAMPION_NAME}  AUC(macro)={CHAMPION_AUC:.4f}")

mlfc=mlflow.tracking.MlflowClient()
try: mlfc.create_registered_model(MLFLOW_MODEL_NAME,
        description="PS3 Root Cause Classification 3-class — Chicago Ventra")
except: pass
mv=mlfc.create_model_version(name=MLFLOW_MODEL_NAME,
    source=f"runs:/{CHAMPION_RUN_ID}/model",run_id=CHAMPION_RUN_ID,
    tags={"algorithm":CHAMPION_NAME,"test_auc_macro":str(CHAMPION_AUC),
          "feasibility":"70pct","target":"failure_level_label(3-class)"})
mlfc.transition_model_version_stage(MLFLOW_MODEL_NAME,mv.version,"Staging")
MLFLOW_VERSION=mv.version
print(f"MLflow: {MLFLOW_MODEL_NAME} v{MLFLOW_VERSION} → Staging")

# SageMaker registration
# Load with the correct MLflow flavor — models were logged with native xgboost/lightgbm/sklearn
_champ_run   = mlfc.get_run(CHAMPION_RUN_ID)
_champ_type  = _champ_run.data.tags.get("model_type", "")
if "xgboost" in _champ_type:
    champion_model = mlflow.xgboost.load_model(f"runs:/{CHAMPION_RUN_ID}/model")
elif "lightgbm" in _champ_type:
    champion_model = mlflow.lightgbm.load_model(f"runs:/{CHAMPION_RUN_ID}/model")
else:  # catboost or nlp pipeline (sklearn flavor)
    champion_model = mlflow.sklearn.load_model(f"runs:/{CHAMPION_RUN_ID}/model")
print(f"Loaded champion ({_champ_type}) via MLflow")
joblib.dump(champion_model,"/tmp/ps3_champion.joblib")
joblib.dump(STRUCT_COLS,"/tmp/ps3_feature_cols.joblib")
test_metrics={"test_auc_macro":CHAMPION_AUC}
S3_CLIENT.put_object(Bucket=ARTIFACT_BUCKET,Key="sagemaker/ps3/metrics/test_metrics.json",
                     Body=json.dumps(test_metrics),ContentType="application/json")
with tarfile.open("/tmp/model.tar.gz","w:gz") as tar:
    tar.add("/tmp/ps3_champion.joblib",arcname="ps3_champion.joblib")
    tar.add("/tmp/ps3_feature_cols.joblib",arcname="ps3_feature_cols.joblib")
MODEL_S3_KEY=f"sagemaker/ps3/model-v{MLFLOW_VERSION}/model.tar.gz"
S3_CLIENT.upload_file("/tmp/model.tar.gz",ARTIFACT_BUCKET,MODEL_S3_KEY)

import sagemaker.image_uris as sm_images
image_uri=sm_images.retrieve(
    framework="sklearn", region=REGION, version="1.2-1",
    instance_type="ml.m5.large", image_scope="inference")
try: SM_CLIENT.create_model_package_group(ModelPackageGroupName=MODEL_REGISTRY_NAME,
        ModelPackageGroupDescription="PS3 Root Cause — Chicago Ventra")
except: pass
resp=SM_CLIENT.create_model_package(
    ModelPackageGroupName=MODEL_REGISTRY_NAME,
    ModelPackageDescription=f"{CHAMPION_NAME}|AUC={CHAMPION_AUC:.4f}|Gv{GOLD_VERSION}",
    InferenceSpecification={
        "Containers":[{"Image":image_uri,"ModelDataUrl":f"s3://{ARTIFACT_BUCKET}/{MODEL_S3_KEY}"}],
        "SupportedContentTypes":["text/csv"],"SupportedResponseMIMETypes":["application/json"],
        "SupportedRealtimeInferenceInstanceTypes":["ml.m5.large"]},
    ModelApprovalStatus="PendingManualApproval",
    CustomerMetadataProperties={"ps":"PS3","algorithm":CHAMPION_NAME,
        "test_auc_macro":str(round(CHAMPION_AUC,4)),"feasibility":"70pct",
        "mlflow_version":str(MLFLOW_VERSION)})
SM_MODEL_PKG_ARN=resp["ModelPackageArn"]
print(f"SM Package: {SM_MODEL_PKG_ARN}")

In [ ]:
# ── CELL 20 : Deploy + Monitors ──────────────────────────────────────────────
from sagemaker.model import ModelPackage
from sagemaker.model_monitor import (DefaultModelMonitor, DataCaptureConfig,
                                      CronExpressionGenerator)
from sagemaker.model_monitor.dataset_format import DatasetFormat

# Clean up any prior endpoint + config with the same name so re-runs don't fail
for _ec_name in [ENDPOINT_NAME]:
    try:
        SM_CLIENT.delete_endpoint(EndpointName=_ec_name)
        print(f"Deleted existing endpoint: {_ec_name}")
        import time; time.sleep(10)
    except Exception: pass
    try:
        SM_CLIENT.delete_endpoint_config(EndpointConfigName=_ec_name)
        print(f"Deleted existing endpoint config: {_ec_name}")
    except Exception: pass

predictor = None
if globals().get("CHAMPION_RUN_ID") is None:
    print("[SKIP] No champion selected — skipping endpoint deploy.")
else:
    try:
        SM_CLIENT.update_model_package(ModelPackageArn=SM_MODEL_PKG_ARN,
                                        ModelApprovalStatus="Approved")
        registered = ModelPackage(role=ROLE, model_package_arn=SM_MODEL_PKG_ARN,
                                  sagemaker_session=SM_SESSION)
        predictor = registered.deploy(
            initial_instance_count=1, instance_type="ml.m5.large",
            endpoint_name=ENDPOINT_NAME,
            data_capture_config=DataCaptureConfig(
                enable_capture=True, sampling_percentage=100,
                destination_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps3/data-capture/",
                capture_options=["Input", "Output"]))
        print(f"Endpoint deployed: {ENDPOINT_NAME}")
    except Exception as _deploy_err:
        print(f"[WARN] Endpoint deploy failed: {_deploy_err}")

if predictor is not None:
    monitor = DefaultModelMonitor(role=ROLE, instance_count=1,
        instance_type="ml.m5.large", sagemaker_session=SM_SESSION)
    import os
    if os.path.exists("/tmp/ps3_baseline.csv"):
        S3_CLIENT.upload_file("/tmp/ps3_baseline.csv", ARTIFACT_BUCKET,
                              "sagemaker/ps3/baseline/baseline.csv")
        monitor.suggest_baseline(
            baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps3/baseline/baseline.csv",
            dataset_format=DatasetFormat.csv(header=True),
            output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps3/baseline/output/",
            wait=True, logs=False)
        monitor.create_monitoring_schedule(
            monitor_schedule_name="chicago-ps3-data-quality",
            endpoint_input=ENDPOINT_NAME,
            output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps3/monitor-output/",
            statistics=monitor.baseline_statistics(),
            constraints=monitor.suggested_constraints(),
            schedule_cron_expression=CronExpressionGenerator.daily(),
            enable_cloudwatch_metrics=True)
        print("Monitor scheduled.")
    else:
        print("[SKIP] /tmp/ps3_baseline.csv not found — monitor not scheduled.")
else:
    print("[SKIP] Monitor not scheduled — endpoint not deployed.")


In [ ]:
# ── CELL 21 : Final Summary ──────────────────────────────────────────────────────────────
_cn  = globals().get("CHAMPION_NAME",  "N/A")
_ca  = globals().get("CHAMPION_AUC",   -1.0)
_mv  = globals().get("MLFLOW_VERSION", "N/A")
# ── CELL 21 : Summary ────────────────────────────────────────────────────────
print("="*60)
print("PS3 — Root Cause Classification — Complete")
print("="*60)
print(f"Champion     : {_cn}  AUC(macro)={_ca:.4f}")
print(f"MLflow model : {MLFLOW_MODEL_NAME} v{_mv}")
print(f"SM endpoint  : {ENDPOINT_NAME}")
print(f"Feature Store: {FS_GROUP_NAME}")
print(f"\nFEASIBILITY = 70%  (updated 2026-06-23, was 55%):")
print("  CONFIRMED: AE_FAILURE_LEVEL Ventra taxonomy (hardware faults, is_device_fault=TRUE):")
print("    1=NONPAYMENT  2=PURCHASE_CARD  3=PURCHASE_PRODUCT")
print("    4=ALL_PURCHASE  5=ALL_FUNCTIONS  16=BUS_READER_ASSEMBLY")
print("  Decoded via silver.dim_failure_level (S18).")
print("")
print("  REMAINING GAP (30%):")
print("  BLOCKED: SVN_STAGE.U_FS_FAULT_CODES (0 rows) → fc_description 9-class BLOCKED")
print("  BLOCKED: SVN_STAGE.U_FS_ACTION_CODES (0 rows) → ac_description 6-class BLOCKED")
print("  WORKAROUND: CTA.SERVICENOW_AVAILABILITY_EVENTS → 3-class failure_level_label ✓")
print("  WORKAROUND: Rule-based root_cause_category (8-class text match) ✓")
print("\nTo reach 100%: SVN_STAGE ETL must be activated (all 35 tables have 0 rows)")

In [ ]:
# ── FINAL CELL : Export notebook with all outputs to HTML ───────────────────
# Permanent fix for SageMaker Studio output stripping.
# Run this once at the end of a full notebook run.
# The HTML file is saved to EFS (/home/sagemaker-user/PS3/) and persists
# across sessions, instance stops, and kernel restarts.
# Open it in any browser to see all cell outputs, tables, and charts.

import subprocess, os, datetime

_nb_path  = f"/home/sagemaker-user/PS3/PS3.ipynb"
_ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M")
_out_html = f"/home/sagemaker-user/PS3/PS3_output_{_ts}.html"

if os.path.exists(_nb_path):
    _result = subprocess.run(
        ["jupyter", "nbconvert", "--to", "html", _nb_path, "--output", _out_html],
        capture_output=True, text=True
    )
    if _result.returncode == 0:
        _size = os.path.getsize(_out_html) / 1024
        print(f"[HTML EXPORT] Saved → {_out_html}  ({_size:.0f} KB)")
        print(f"  Open in browser — all cell outputs preserved permanently.")
    else:
        print(f"[HTML EXPORT] nbconvert error:")
        print(_result.stderr[-500:] if _result.stderr else "(no stderr)")
        # Fallback: save key outputs to a text log instead
        _log = f"/home/sagemaker-user/PS3/PS3_run_{_ts}.log"
        print(f"  Fallback: write key results to {_log} manually.")
else:
    print(f"[HTML EXPORT] Notebook not found at {_nb_path}")
    print(f"  Actual path on EFS may differ — update _nb_path above.")
    print(f"  Run: !find /home/sagemaker-user -name 'PS3.ipynb' to locate it.")
